# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AviJxn/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os, sys, subprocess, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# 1. Environment & Path Handling
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
        if os.path.exists("flyrank-ml-internship"):
            os.chdir("flyrank-ml-internship")
        else:
            REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-ml-internship"], check=True)
            os.chdir("flyrank-ml-internship")

data_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    for level in ["..", "../..", "../../.."]:
        test_path = os.path.join(level, data_path)
        if os.path.exists(test_path):
            data_path = test_path
            break

df = pd.read_csv(data_path)
df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)

# 2. Fit Validated Week-5 Model to Generate Action Probabilities
feature_cols = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]
X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining"].values

rf = RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=10, random_state=42, n_jobs=-1)
rf.fit(X, y)
df["decay_probability"] = rf.predict_proba(X)[:, 1]

# 3. Archetype & Reason Code Mapping
def map_action_archetype(row):
    prob = row["decay_probability"]
    staleness = row["days_since_last_update"]
    ctr = row["ctr"]
    position = row["avg_position"]
    impressions = row["impressions_90d"]

    if prob >= 0.65 and staleness >= 180 and impressions >= 500:
        return pd.Series({
            "reason_code": "HIGH_EXPOSURE_STALE",
            "recommended_action": "FULL_EDITORIAL_REFRESH",
            "priority_tier": "P1_URGENT",
            "estimated_cost_usd": 350
        })
    elif prob >= 0.50 and position <= 15.0 and ctr < 0.02:
        return pd.Series({
            "reason_code": "CTR_UNDERPERFORMER",
            "recommended_action": "TITLE_META_OPTIMIZATION",
            "priority_tier": "P2_HIGH",
            "estimated_cost_usd": 50
        })
    elif prob >= 0.50 and impressions < 100:
        return pd.Series({
            "reason_code": "LOW_TRAFFIC_STAGNANT",
            "recommended_action": "PRUNE_OR_CONSOLIDATE",
            "priority_tier": "P3_MEDIUM",
            "estimated_cost_usd": 100
        })
    else:
        return pd.Series({
            "reason_code": "STABLE_PERFORMER",
            "recommended_action": "MONITOR",
            "priority_tier": "P4_LOW",
            "estimated_cost_usd": 0
        })

playbook_meta = df.apply(map_action_archetype, axis=1)
df = pd.concat([df, playbook_meta], axis=1)

# Sort Action Queue by Probability
action_queue = df.sort_values("decay_probability", ascending=False).reset_index(drop=True)

print("=== Content Action Playbook Queue Summary ===")
print(action_queue["recommended_action"].value_counts())
display(action_queue[["decay_probability", "priority_tier", "reason_code", "recommended_action", "days_since_last_update", "impressions_90d", "ctr"]].head(10))

=== Content Action Playbook Queue Summary ===
recommended_action
MONITOR                    24004
TITLE_META_OPTIMIZATION     4696
PRUNE_OR_CONSOLIDATE        1290
FULL_EDITORIAL_REFRESH        10
Name: count, dtype: int64


,decay_probability,priority_tier,reason_code,recommended_action,days_since_last_update,impressions_90d,ctr
0,0.764670,P4_LOW,STABLE_PERFORMER,MONITOR,104,2748,0.07
1,0.764568,P4_LOW,STABLE_PERFORMER,MONITOR,104,2087,0.05
2,0.764300,P4_LOW,STABLE_PERFORMER,MONITOR,104,2765,0.04
3,0.763557,P4_LOW,STABLE_PERFORMER,MONITOR,104,1076,0.09
4,0.762985,P4_LOW,STABLE_PERFORMER,MONITOR,104,748,0.13
5,0.762985,P4_LOW,STABLE_PERFORMER,MONITOR,104,3382,0.12
6,0.762578,P4_LOW,STABLE_PERFORMER,MONITOR,104,894,0.11
7,0.762578,P4_LOW,STABLE_PERFORMER,MONITOR,104,704,0.14
8,0.762578,P4_LOW,STABLE_PERFORMER,MONITOR,104,835,0.12
9,0.762578,P4_LOW,STABLE_PERFORMER,MONITOR,104,908,0.11


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


### Intended Use
* **Primary Scope**: Weekly content review queue for B2B/B2C organic search teams.
* **Operational Role**: Decision-support tool to rank articles needing human editorial evaluation.
* **Target Audience**: Content Strategists, SEO Specialists, and Managing Editors.

### Operational Boundaries & Non-Goals
* **Not Automated Publishing**: The model does NOT automatically write, edit, or publish content updates without human oversight.
* **Not Causal Assurance**: A high decay probability indicates historical performance loss; it does NOT guarantee that rewriting a page will restore rankings if Google SERP layout changes occur.
* **Domain Context**: Calibrated for informational blog posts and documentation; inapplicable to dynamic product inventory pages.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*


### Mandatory Human Review Checklist
1. **Search Intent Verification**: Confirm Google SERP layout has not shifted from text-based articles to video, tools, or local packs.
2. **Technical Health Audit**: Verify that traffic loss is not caused by broken canonical tags, 404 errors, or indexing blocks.
3. **Factual Accuracy Check**: Ensure outdated statistics, prices, or product names are manually updated by a Subject Matter Expert (SME).

### Strict No-Go List (NEVER Automate)
* **Legal & Compliance Pages**: Privacy policies, Terms of Service, and regulatory disclosure pages.
* **Core Brand Homepages & Conversion Landing Pages**: Key transactional pages where traffic drops require conversion rate optimization (CRO) audits, not text refreshes.
* **Recently Migrated URLs**: Pages redirected within the last 60 days where GSC reporting metrics remain volatile.
* **Mass LLM Auto-Rewriting**: Generating fully automated AI text updates without human editor sign-off.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


### Performance Degradation Triggers
* **Precision Decay**: Retrain if **Precision@50** drops below `0.50` on weekly holdout validation checks.
* **Class Imbalance Drift**: Retrain if the global decay rate shifts by more than `15%` relative to baseline historical distributions.

### Data Drift Triggers (PSI & Feature Shift)
* **Population Stability Index (PSI)**: Monitor distribution drift on `impressions_90d` and `ctr`. Trigger model re-calibration if $PSI > 0.25$.
* **Core Update Events**: Trigger immediate retrain and feature re-scaling following confirmed Google Broad Core Updates.
* **Scheduled Cadence**: Re-estimate feature importances monthly; full retrain every 90 days.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
# Create export directories
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Export Action Queue CSV
queue_export_path = "work/outputs/action_playbook_queue.csv"
export_cols = [
    "decay_probability",
    "priority_tier",
    "reason_code",
    "recommended_action",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "estimated_cost_usd"
]
action_queue[export_cols].to_csv(queue_export_path, index=False)
print(f"Action Queue exported to '{queue_export_path}' ({len(action_queue):,} rows).")

# 2. Export Metrics JSON
metrics = {
    "total_actionable_items": int((df["decay_probability"] >= 0.50).sum()),
    "p1_urgent_count": int((df["priority_tier"] == "P1_URGENT").sum()),
    "p2_high_count": int((df["priority_tier"] == "P2_HIGH").sum()),
    "total_estimated_budget_usd": float(df["estimated_cost_usd"].sum()),
    "recommended_weekly_capacity": 20
}
metrics_json_path = "work/outputs/playbook_metrics.json"
with open(metrics_json_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics JSON exported to '{metrics_json_path}':")
print(json.dumps(metrics, indent=2))

# 3. Export Visual Plot for Paper
plt.figure(figsize=(8, 4.5))
df["recommended_action"].value_counts().plot(kind="bar", color="#2b5c8f", edgecolor="black")
plt.title("Content Action Playbook Queue Distribution")
plt.xlabel("Recommended Action")
plt.ylabel("Number of Content Items")
plt.xticks(rotation=15)
plt.tight_layout()

figure_path = "work/figures/action_distribution.png"
plt.savefig(figure_path, dpi=300)
plt.close()

print(f"Plot saved to '{figure_path}'.")

Action Queue exported to 'work/outputs/action_playbook_queue.csv' (30,000 rows).
Metrics JSON exported to 'work/outputs/playbook_metrics.json':
{
  "total_actionable_items": 20873,
  "p1_urgent_count": 10,
  "p2_high_count": 4696,
  "total_estimated_budget_usd": 367300.0,
  "recommended_weekly_capacity": 20
}
Plot saved to 'work/figures/action_distribution.png'.


## 6. Self-Check

- [x] **Ranked Actions & Reason Codes**: Mapped model probabilities into actionable archetypes (`HIGH_EXPOSURE_STALE`, `CTR_UNDERPERFORMER`, `LOW_TRAFFIC_STAGNANT`).
- [x] **Intended Use & Limits Defined**: Stated clear decision-support scope and operational boundaries.
- [x] **No-Go List Established**: Defined strict rules barring automated rewrites, legal pages, and transactional landing pages.
- [x] **Monitoring & Retrain Triggers**: Set explicit Precision@50 thresholds ($<0.50$) and PSI drift triggers ($>0.25$).
- [x] **Artifacts Exported**: Written `work/outputs/action_playbook_queue.csv`, `work/outputs/playbook_metrics.json`, and `work/figures/action_distribution.png`.